# Experiment 3 — Jev Routing with LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipundavid/ai-experiments/blob/main/jev/src/experiment_3_routing.ipynb)

This notebook reproduces the `experiment_3_routing.py` workflow: Jev classifies a query into a category, then LangGraph routes execution to the matching worker.

> Set `TYPESAFE_API_KEY` before running this notebook.

## Overview

This experiment shows the routing pattern:

- Jev decides the category
- LangGraph routes to the relevant worker
- each worker returns a result for the selected path

The available categories include a fallback for insufficient evidence.

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if 'TYPESAFE_API_KEY' not in os.environ:
    raise RuntimeError('Set the TYPESAFE_API_KEY environment variable before running this notebook.')

In [ ]:
from typing import Literal, TypedDict

from langchain_typesafe import Choice, TypeSafeClassifier
from langgraph.graph import END, START, StateGraph

classifier = TypeSafeClassifier(api_key=os.environ['TYPESAFE_API_KEY'])
Category = Literal['rag', 'coding', 'system_design', 'general', 'insufficient_evidence']

class State(TypedDict, total=False):
    query: str
    category: Category
    result: str


def classify_query(state: State):
    response = classifier.invoke(
        {
            'state': state['query'],
            'questions': {
                'category': Choice(
                    instructions='Classify the query into exactly one category.',
                    criteria={
                        'rag': 'RAG, retrieval, embeddings, or vector databases.',
                        'coding': 'Writing, debugging, or understanding code.',
                        'system_design': 'Software architecture or system design.',
                        'general': 'Anything that does not fit the other categories.',
                        'insufficient_evidence': 'The query does not contain enough information to assign a reliable category.',
                    },
                )
            },
        }
    )
    return {'category': response.choices['category'].choice}


def route_query(state: State) -> Category:
    return state['category']


def make_worker(label: str):
    def worker(state: State):
        return {'result': f'{label} worker selected for: {state["query"]}'}

    return worker


graph = StateGraph(State)
graph.add_node('classify_query', classify_query)
for category in ('rag', 'coding', 'system_design', 'general', 'insufficient_evidence'):
    graph.add_node(category, make_worker(category))

graph.add_edge(START, 'classify_query')
graph.add_conditional_edges(
    'classify_query',
    route_query,
    {
        category: category
        for category in ('rag', 'coding', 'system_design', 'general', 'insufficient_evidence')
    },
)
for category in ('rag', 'coding', 'system_design', 'general', 'insufficient_evidence'):
    graph.add_edge(category, END)

app = graph.compile()

In [ ]:
query = 'How should I design a RAG pipeline?'
result = app.invoke({'query': query})

print('\n' + '=' * 70)
print('FINAL LANGGRAPH STATE')
print('=' * 70)
print(f"Category: {result['category']}")
print(result['result'])

## Result interpretation

This notebook illustrates one of the main benefits of Jev in an agent workflow: a structured category decision can directly control branching without requiring the application to parse free-form LLM output.